# U11 练习题 | Bahdanau Attention

过关标准：

1. 跑通 Hugging Face 小批量数据或 fallback 数据。
2. 补全 `BahdanauAttention`，确认 attention 权重每行求和接近 1。
3. 补全 `AttnDecoder` 和 `Seq2SeqAttention`。
4. 训练至少 5 轮，并能画出一张 attention 热力图。

建议：先独立写，卡住后再回 lesson 对照。

## 练习 11.1：数据加载 + 专业分词器

目标：

- 使用 Hugging Face `datasets` 加载小批量中英数据。
- 中文用 `jieba`，英文用 Moses tokenizer。
- 如果 HF 下载失败，使用 fallback 数据。

填空重点：`tokenize_zh`、`tokenize_en`、`extract_zh_en`。

In [ ]:
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import jieba
from sacremoses import MosesTokenizer
from datasets import load_dataset

PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']

EMBED_DIM = 64
ENC_HIDDEN_DIM = 64
DEC_HIDDEN_DIM = 128
BATCH_SIZE = 16
MAX_PAIRS = 128
LR = 1e-3
EPOCHS = 5

random.seed(0)
torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

moses_en = MosesTokenizer(lang='en')


def normalize_text(text):
    return str(text).replace('\u3000', ' ').strip()


def tokenize_zh(text):
    # TODO 1: 如果文本里已经有空格，按空格切；否则用 jieba.lcut
    pass


def tokenize_en(text):
    # TODO 2: 小写后用 MosesTokenizer 分词，返回 list[str]
    pass


def extract_zh_en(example):
    # TODO 3: 优先处理 example['translation'] = {'zh': ..., 'en': ...} 的情况
    # 返回 (zh, en)，失败返回 None
    pass


def load_translation_pairs_from_hf(max_pairs=MAX_PAIRS):
    ds = load_dataset('Helsinki-NLP/opus-100', 'en-zh', split=f'train[:{max_pairs * 4}]')
    pairs = []
    for ex in ds:
        item = extract_zh_en(ex)
        if item is None:
            continue
        zh, en = item
        if zh and en:
            pairs.append((zh, en.lower()))
        if len(pairs) >= max_pairs:
            break
    return pairs


fallback_pairs = [
    ('我 爱 你', 'i love you'),
    ('谢谢 你', 'thank you'),
    ('猫 在 睡觉', 'the cat is sleeping'),
    ('狗 在 跑步', 'the dog is running'),
]

try:
    raw_pairs = load_translation_pairs_from_hf(MAX_PAIRS)
except Exception as e:
    print('HF 数据加载失败，使用 fallback:', repr(e))
    raw_pairs = fallback_pairs

print('pairs:', len(raw_pairs))
print(raw_pairs[:2])
print(tokenize_zh(raw_pairs[0][0]))
print(tokenize_en(raw_pairs[0][1]))

## 练习 11.2：Vocab、Dataset、collate_fn

目标：复用 U09/U10 的数据管道，但这次 token 来自专业分词器。

In [ ]:
class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)
        self.itos = list(SPECIALS)
        for token, freq in counter.most_common():
            if freq >= min_freq and token not in self.itos:
                self.itos.append(token)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(token, UNK) for token in tokens]

    def decode(self, ids):
        return [self.itos[int(i)] for i in ids]


def sentence_to_ids(sentence, vocab, tokenizer, add_sos=False, add_eos=True):
    # TODO 1: 分词 -> encode -> 按需加 SOS/EOS
    pass


def pad_sequence(ids_list, pad_id=PAD):
    # TODO 2: padding 到 batch 内最长，返回 LongTensor
    pass


src_vocab = Vocab([tokenize_zh(zh) for zh, en in raw_pairs])
tgt_vocab = Vocab([tokenize_en(en) for zh, en in raw_pairs])


class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # TODO 3: 返回 src_ids, tgt_ids
        pass


def collate_fn(batch):
    # TODO 4: 按 src 长度降序，返回 src, src_len, tgt
    pass


loader = DataLoader(TranslationDataset(raw_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
src, src_len, tgt = next(iter(loader))
print(src.shape, src_len.shape, tgt.shape)

## 练习 11.3：Encoder 输出每个源位置

目标：双向 GRU 返回：

- `encoder_outputs: (B,S,2H_enc)`
- `decoder_init: (1,B,H_dec)`

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        # TODO 1: embedding
        # TODO 2: bidirectional GRU
        # TODO 3: bridge Linear，把 2H_enc -> H_dec
        pass

    def forward(self, src, src_len):
        # TODO 4: embedding -> pack -> gru -> unpack
        # TODO 5: 拼接最后一层 forward/backward hidden，过 bridge 得到 decoder_init
        pass


enc = Encoder(len(src_vocab), EMBED_DIM, ENC_HIDDEN_DIM, DEC_HIDDEN_DIM)
enc_out, enc_hidden = enc(src, src_len)
print(enc_out.shape)
print(enc_hidden.shape)

## 练习 11.4：补全 BahdanauAttention

目标：实现：

$$
e_{t,i} = \mathbf{v}_a^\top \tanh\left(\mathbf{W}_e \mathbf{h}^{enc}_i + \mathbf{W}_d \mathbf{s}_{t-1}\right)
$$

$$
\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{j=1}^{S}\exp(e_{t,j})}
$$

$$
\mathbf{c}_t = \sum_{i=1}^{S} \alpha_{t,i}\mathbf{h}^{enc}_i
$$

代码实现时对应四步：`score -> mask PAD -> softmax -> bmm`。

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, enc_output_dim, dec_hidden_dim):
        super().__init__()
        # TODO 1: W_enc, W_dec, v 三个 Linear
        pass

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        # decoder_hidden: (1,B,H_dec)
        # encoder_outputs: (B,S,2H_enc)
        # src_mask: (B,S), True 表示真实 token
        # TODO 2: 取 decoder_hidden[-1] 并 unsqueeze 成 query
        # TODO 3: 计算 scores: (B,S)
        # TODO 4: mask PAD 位置
        # TODO 5: softmax 得 attn_weights
        # TODO 6: bmm 得 context
        pass


attn = BahdanauAttention(ENC_HIDDEN_DIM * 2, DEC_HIDDEN_DIM)
src_mask = src.ne(PAD)
context, attn_weights = attn(enc_hidden, enc_out, src_mask)
print(context.shape)
print(attn_weights.shape)
print(attn_weights.sum(dim=1)[:5])

## 练习 11.5：补全 AttnDecoder + Seq2SeqAttention

目标：把 attention 接入 decoder，并返回 `outputs` 和 `attentions`。

In [ ]:
class AttnDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_output_dim, dec_hidden_dim):
        super().__init__()
        # TODO 1: embedding, attention, gru, fc
        pass

    def forward(self, x, hidden, encoder_outputs, src_mask):
        # TODO 2: embedding
        # TODO 3: attention 得 context
        # TODO 4: concat embedding + context 送 GRU
        # TODO 5: concat output + context + embedding 送 fc
        pass


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        # TODO 6: 参考 lesson，循环解码并保存 outputs/attentions
        pass

## 练习 11.6：训练 + 推理 + 热力图

目标：跑通 5 轮训练，并实现 `translate_with_attention`。

In [ ]:
encoder = Encoder(len(src_vocab), EMBED_DIM, ENC_HIDDEN_DIM, DEC_HIDDEN_DIM)
decoder = AttnDecoder(len(tgt_vocab), EMBED_DIM, ENC_HIDDEN_DIM * 2, DEC_HIDDEN_DIM)
model = Seq2SeqAttention(encoder, decoder).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
V_tgt = len(tgt_vocab)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n_steps = 0.0, 0
    for src, src_len, tgt in loader:
        # TODO 1: to(device), zero_grad, forward, loss, backward, clip_grad, step
        pass
    print(f'Epoch {epoch:3d} | loss={total_loss / max(n_steps, 1):.4f}')


def translate_with_attention(model, sentence, max_len=30):
    # TODO 2: greedy decode，同时保存每一步 attn_weights
    pass


for zh, en in raw_pairs[:5]:
    pred, attn_matrix, src_tokens, pred_tokens = translate_with_attention(model, zh)
    print(f'{zh} -> {pred}   (gold: {en})')

## 练习 11.7：口头/笔头回答

1. U10 的 Decoder 为什么只需要 `hidden`，U11 为什么还需要 `encoder_outputs`？
2. `attn_weights` 的 shape 是 `(B,S)`，为什么不是 `(B,T,S)`？
3. `Seq2SeqAttention` 返回的 `attentions` 为什么是 `(B,T,S)`？
4. 为什么 attention 里必须 mask PAD？
5. `torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)` 中，两个矩阵乘法输入的 shape 分别是什么？
6. 专业分词器相比中文按字切分，解决了什么问题？